# 01 · Exploración del login en saucedemo

Notebook interactivo para recorrer el login paso a paso con Selenium y los Page Objects del proyecto.

- Chrome se abre en modo visible (para verlo en headless: variable de entorno `HEADLESS=1`).
- Las mismas validaciones están automatizadas en `tests/test_login.py`.

In [ ]:
import sys
from pathlib import Path

# Permite importar pages/ y utils/ desde la raíz del proyecto
sys.path.insert(0, str(Path.cwd().parent))

from IPython.display import Image, display

from pages import LoginPage, InventoryPage
from utils.config import USERS, PASSWORD
from utils.driver_factory import create_driver

def captura(page, width=700):
    display(Image(page.screenshot_png(), width=width))

## 1. Abrir el navegador y la página de login

In [ ]:
driver = create_driver()
login_page = LoginPage(driver).load()
print(driver.title, '->', login_page.current_url)
captura(login_page)

## 2. Casos negativos

Saucedemo tiene usuarios pensados para probar errores. Recorremos cada caso y mostramos el mensaje que entrega el sitio.

In [ ]:
casos_negativos = [
    ('Usuario bloqueado', USERS['locked_out'], PASSWORD),
    ('Password incorrecta', USERS['standard'], 'clave_incorrecta'),
    ('Sin usuario', '', PASSWORD),
    ('Sin password', USERS['standard'], ''),
]

for descripcion, usuario, clave in casos_negativos:
    login_page.load().login(usuario, clave)
    print(f'{descripcion:<20} -> {login_page.error_message()}')

In [ ]:
captura(login_page)

## 3. Login exitoso con `standard_user`

In [ ]:
inventory = login_page.load().login_as(USERS['standard'])
print('Título:', inventory.title())
print('URL:', inventory.current_url)
captura(inventory)

## 4. Logout

In [ ]:
login_page = inventory.logout()
print('¿Volvió al login?', login_page.is_displayed())

## Cierre

In [ ]:
driver.quit()